# Validation Graph Explorer

Use this notebook to locate a validation/test pickle dumped by `Trainer.run_validation` and view the stored predicted graphs in interactive 3D.

### Prerequisites
- Ensure your validation runs saved pickles under `outputs/.../validation/step_*.pkl`.
- Install the optional dependencies used here (if they are not already in your environment):
  ```bash
  pip install plotly ipywidgets
  ```
- Enable ipywidgets extensions in JupyterLab if prompted (e.g., `pip install jupyterlab_widgets`).

Run the cells below top-to-bottom, then use the dropdowns/sliders to pick a saved pickle, EMA beta, graph index, and depth cutoff.

In [ ]:
from pathlib import Path
import pickle

import networkx as nx
import numpy as np
import plotly.graph_objects as go
from ipywidgets import Dropdown, IntSlider, HTML, Output, VBox
from IPython.display import display, clear_output

# Notebook lives in /notebooks; add direct parent (repo root) for imports.
import sys
repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))
from utils.data_loading import load_swc_graphs_from_dir


print("Imports ready.")

Imports ready.


In [ ]:
# Root directory that Hydra/Trainer writes into. Adjust if you keep outputs elsewhere.
# OUTPUT_ROOT = Path('../full_run_outs').resolve()
OUTPUT_ROOT = Path('/home/mari350i/ws/tree_generation/outputs/2026-04-15/19-15-55/validation').resolve()  # bigger model
# OUTPUT_ROOT = Path('/home/mari350i/ws/tree_generation/outputs/2026-04-15/19-15-59/validation').resolve()  # small model

# Ground-truth dataset root used for same-index pairing against pred_graphs.
GT_ROOT = Path('../data/small_trees/val').resolve()


def step_sort_key(path: Path):
    stem = path.stem  # e.g. step_95000
    if stem.startswith('step_'):
        suffix = stem.split('step_', 1)[1]
        if suffix.isdigit():
            return (0, int(suffix))
    # Keep non-standard files after step_<n>.pkl, deterministic by name.
    return (1, stem)


available_pickles = sorted(OUTPUT_ROOT.glob('*.pkl'), key=step_sort_key)
if not available_pickles:
    raise FileNotFoundError(f'No validation pickles found under {OUTPUT_ROOT}. Run validation first.')

# Load GT once so downstream cells can reuse it.
try:
    gt_graphs = load_swc_graphs_from_dir(GT_ROOT)
    gt_load_error = None
except Exception as exc:
    gt_graphs = []
    gt_load_error = str(exc)

# Count check: compare GT count with pred_graphs count in latest pickle per EMA beta.
with open(available_pickles[-1], 'rb') as f:
    latest_payload = pickle.load(f)
pred_counts = {
    beta_key: len(beta_payload.get('pred_graphs', []))
    for beta_key, beta_payload in latest_payload.items()
    if isinstance(beta_payload, dict)
}

print(f'Discovered {len(available_pickles)} pickle file(s). Latest: {available_pickles[-1]}')
if gt_load_error:
    print(f'GT load failed from {GT_ROOT}: {gt_load_error}')
else:
    print(f'Loaded {len(gt_graphs)} GT graph(s) from {GT_ROOT}.')
    if pred_counts:
        mismatched = {k: v for k, v in pred_counts.items() if v != len(gt_graphs)}
        if mismatched:
            print('Count mismatch detected (pred count vs GT count):')
            for beta_key, pred_n in mismatched.items():
                print(f'  {beta_key}: pred={pred_n}, gt={len(gt_graphs)}')
        else:
            print('Count check passed for latest pickle: pred_graphs count matches GT count for all EMA betas.')

Discovered 20 pickle file(s). Latest: /data/horse/ws/mari350i-tree_generation/outputs/2026-04-15/19-15-55/validation/step_100000.pkl
Loaded 50 GT graph(s) from /data/horse/ws/mari350i-tree_generation/data/small_trees/val.
Count check passed for latest pickle: pred_graphs count matches GT count for all EMA betas.


In [ ]:
def load_validation_pickle(path: Path):
    path = Path(path)
    with open(path, 'rb') as f:
        data = pickle.load(f)
    return data


def build_plotly_figure(
    graph: nx.Graph,
    title: str,
    node_color: str = 'royalblue',
    edge_color: str = 'lightgray',
    camera_eye: dict | None = None,
):
    if camera_eye is None:
        camera_eye = {'x': 1.8, 'y': 1.8, 'z': 1.8}

    positions = {}
    for node in graph.nodes():
        pos = np.asarray(graph.nodes[node].get('pos', np.zeros(3)), dtype=float).flatten()
        if pos.size < 3:
            pos = np.pad(pos, (0, 3 - pos.size), constant_values=0.0)
        positions[node] = pos[:3]

    node_x, node_y, node_z, node_labels = [], [], [], []
    for node, pos in positions.items():
        node_x.append(pos[0])
        node_y.append(pos[1])
        node_z.append(pos[2])
        node_labels.append(str(node))

    edge_x, edge_y, edge_z = [], [], []
    for u, v in graph.edges():
        x0, y0, z0 = positions[u]
        x1, y1, z1 = positions[v]
        edge_x.extend([x0, x1, None])
        edge_y.extend([y0, y1, None])
        edge_z.extend([z0, z1, None])

    fig = go.Figure()
    if edge_x:
        fig.add_trace(
            go.Scatter3d(
                x=edge_x,
                y=edge_y,
                z=edge_z,
                mode='lines',
                line=dict(color=edge_color, width=2),
                hoverinfo='skip',
                name='edges',
                showlegend=False,
            )
        )

    fig.add_trace(
        go.Scatter3d(
            x=node_x,
            y=node_y,
            z=node_z,
            mode='markers',
            marker=dict(size=6, color=node_color, line=dict(color='black', width=0.5)),
            text=node_labels,
            name='nodes',
            showlegend=False,
        )
    )

    fig.update_layout(
        title=title,
        showlegend=False,
        scene=dict(
            xaxis_title='x',
            yaxis_title='y',
            zaxis_title='z',
            aspectmode='data',
            camera=dict(eye=camera_eye),
        ),
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


def pretty_label(path: Path):
    path = Path(path)
    try:
        rel = path.relative_to(OUTPUT_ROOT)
        return str(rel)
    except ValueError:
        return str(path)


def compute_node_depths(graph: nx.Graph):
    nodes = list(graph.nodes())
    if not nodes:
        return {}, 0, None
    # Use node 0 as canonical root (matches generation); fallback to first available node.
    root = 0 if 0 in graph else nodes[0]
    lengths = nx.single_source_shortest_path_length(graph, root)
    if lengths:
        missing_depth = max(lengths.values()) + 1
    else:
        missing_depth = 0
    depths = {n: lengths.get(n, missing_depth) for n in nodes}
    max_depth = max(depths.values(), default=0)
    return depths, max_depth, root

In [8]:
from ipywidgets import HBox, Layout

# --- Interactive widgets ---
if 'gt_graphs' not in globals() or 'gt_load_error' not in globals():
    gt_graphs = []
    gt_load_error = 'GT state not initialized. Run Cell 4 first.'

pickle_options = [(pretty_label(p), str(p)) for p in available_pickles]
file_dropdown = Dropdown(
    options=pickle_options,
    description='Result',
    value=pickle_options[-1][1],
    layout=dict(width='70%'),
)
beta_dropdown = Dropdown(options=[], description='EMA beta', layout=dict(width='40%'))
graph_slider = IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description='Graph idx',
    continuous_update=False,
    layout=dict(width='60%'),
)
depth_slider = IntSlider(
    value=0,
    min=0,
    max=0,
    step=1,
    description='Depth',
    continuous_update=False,
    layout=dict(width='60%'),
    disabled=True,
)
status_html = HTML(value='Select a pickle to begin.')
gt_plot_output = Output(layout=Layout(width='50%'))
pred_plot_output = Output(layout=Layout(width='50%'))

validation_payload = load_validation_pickle(file_dropdown.value)

current_pred_depths = {}
current_pred_max_depth = 0
current_pred_root = None
current_gt_depths = {}
current_gt_max_depth = 0
current_gt_root = None
suppress_graph_event = False
suppress_depth_event = False


def clear_depth_state():
    global current_pred_depths, current_pred_max_depth, current_pred_root
    global current_gt_depths, current_gt_max_depth, current_gt_root, suppress_depth_event
    current_pred_depths = {}
    current_pred_max_depth = 0
    current_pred_root = None
    current_gt_depths = {}
    current_gt_max_depth = 0
    current_gt_root = None
    suppress_depth_event = True
    depth_slider.disabled = True
    depth_slider.max = 0
    depth_slider.value = 0
    depth_slider.description = 'Depth'
    suppress_depth_event = False


def set_graph_slider(value: int):
    global suppress_graph_event
    suppress_graph_event = True
    graph_slider.value = value
    suppress_graph_event = False


def set_depth_slider(value: int):
    global suppress_depth_event
    suppress_depth_event = True
    depth_slider.value = value
    suppress_depth_event = False


def clear_plot_outputs(msg_gt: str = '', msg_pred: str = ''):
    with gt_plot_output:
        clear_output(wait=True)
        if msg_gt:
            print(msg_gt)
    with pred_plot_output:
        clear_output(wait=True)
        if msg_pred:
            print(msg_pred)


def refresh_beta_dropdown():
    clear_depth_state()
    betas = sorted(validation_payload.keys())
    if not betas:
        beta_dropdown.options = []
        beta_dropdown.value = None
        graph_slider.disabled = True
        status_html.value = 'Selected pickle contains no EMA entries.'
        clear_plot_outputs('No GT data to display.', 'No data to display for this pickle.')
        return
    beta_dropdown.options = betas
    if beta_dropdown.value not in betas:
        beta_dropdown.value = betas[0]
    refresh_graph_slider()


def refresh_graph_slider():
    beta_key = beta_dropdown.value
    graphs = validation_payload.get(beta_key, {}).get('pred_graphs', []) if beta_key else []
    has_graphs = len(graphs) > 0
    graph_slider.disabled = not has_graphs
    graph_slider.max = max(0, len(graphs) - 1)
    if graph_slider.value > graph_slider.max:
        set_graph_slider(graph_slider.max)
    if has_graphs:
        if gt_load_error:
            gt_msg = f'GT unavailable ({gt_load_error}). Showing predictions only.'
        else:
            gt_msg = f'GT loaded: {len(gt_graphs)} graph(s) from {GT_ROOT}.'
        status_html.value = f"{len(graphs)} generated graph(s) available for {beta_key}. {gt_msg}"
        set_graph_slider(graph_slider.max)
        update_plot(recompute_depth=True)
    else:
        clear_depth_state()
        status_html.value = f'No `pred_graphs` stored for {beta_key}.'
        clear_plot_outputs('No GT graph to display.', 'No graphs to display for this beta selection.')


def update_plot(recompute_depth: bool):
    beta_key = beta_dropdown.value
    pred_graphs = validation_payload.get(beta_key, {}).get('pred_graphs', []) if beta_key else []
    if not pred_graphs:
        return

    idx = int(np.clip(graph_slider.value, 0, len(pred_graphs) - 1))
    pred_graph = pred_graphs[idx]
    gt_graph = gt_graphs[idx] if (not gt_load_error and idx < len(gt_graphs)) else None

    global current_pred_depths, current_pred_max_depth, current_pred_root
    global current_gt_depths, current_gt_max_depth, current_gt_root
    if recompute_depth or not current_pred_depths:
        pred_depths, pred_max_depth, pred_root = compute_node_depths(pred_graph)
        current_pred_depths = pred_depths
        current_pred_max_depth = pred_max_depth
        current_pred_root = pred_root

        if gt_graph is not None:
            gt_depths, gt_max_depth, gt_root = compute_node_depths(gt_graph)
            current_gt_depths = gt_depths
            current_gt_max_depth = gt_max_depth
            current_gt_root = gt_root
        else:
            current_gt_depths = {}
            current_gt_max_depth = 0
            current_gt_root = None

        depth_slider.min = 0
        depth_slider.max = max(current_pred_max_depth, current_gt_max_depth)
        depth_slider.step = 1
        depth_slider.disabled = False
        depth_slider.description = f'Depth (pred root={pred_root})' if pred_root is not None else 'Depth'
        set_depth_slider(depth_slider.max)

    depth_limit = depth_slider.value if not depth_slider.disabled else max(current_pred_max_depth, current_gt_max_depth)

    pred_visible_nodes = [n for n, d in current_pred_depths.items() if d <= depth_limit]
    pred_subgraph = pred_graph.subgraph(pred_visible_nodes).copy()
    pred_hidden_nodes = pred_graph.number_of_nodes() - pred_subgraph.number_of_nodes()

    with pred_plot_output:
        clear_output(wait=True)
        if pred_subgraph.number_of_nodes() == 0:
            print('Prediction: no nodes visible at this depth. Increase the slider.')
        else:
            pred_title = (
                f'{beta_key} | Pred graph {idx} | nodes={pred_subgraph.number_of_nodes()} '
                f'(hidden {pred_hidden_nodes})'
            )
            pred_fig = build_plotly_figure(
                pred_subgraph,
                pred_title,
                node_color='royalblue',
                edge_color='lightgray',
            )
            display(pred_fig)

    gt_hidden_nodes = None
    if gt_graph is not None:
        gt_visible_nodes = [n for n, d in current_gt_depths.items() if d <= depth_limit]
        gt_subgraph = gt_graph.subgraph(gt_visible_nodes).copy()
        gt_hidden_nodes = gt_graph.number_of_nodes() - gt_subgraph.number_of_nodes()
        with gt_plot_output:
            clear_output(wait=True)
            if gt_subgraph.number_of_nodes() == 0:
                print('GT: no nodes visible at this depth. Increase the slider.')
            else:
                gt_title = (
                    f'GT graph {idx} | nodes={gt_subgraph.number_of_nodes()} '
                    f'(hidden {gt_hidden_nodes})'
                )
                gt_fig = build_plotly_figure(
                    gt_subgraph,
                    gt_title,
                    node_color='forestgreen',
                    edge_color='darkseagreen',
                )
                display(gt_fig)
    else:
        with gt_plot_output:
            clear_output(wait=True)
            if gt_load_error:
                print(f'GT unavailable: {gt_load_error}')
            else:
                print(
                    f'No GT graph for index {idx}. Pred graphs: {len(pred_graphs)} | GT graphs: {len(gt_graphs)}.'
                )

    if gt_hidden_nodes is None:
        gt_extra = 'GT unavailable for current index.'
    else:
        gt_extra = f'GT hidden beyond depth {depth_limit}: {gt_hidden_nodes}' if gt_hidden_nodes else 'GT all nodes visible.'
    pred_extra = (
        f'Pred hidden beyond depth {depth_limit}: {pred_hidden_nodes}'
        if pred_hidden_nodes
        else 'Pred all nodes visible.'
    )
    status_html.value = (
        f'{len(pred_graphs)} generated graph(s) available for {beta_key}. '
        f'Pairing: same index with GT from deterministic dataset order. '
        f'{gt_extra} {pred_extra}'
    )


def handle_file_change(change):
    global validation_payload
    new_path = change.get('new')
    if not new_path:
        return
    validation_payload = load_validation_pickle(new_path)
    refresh_beta_dropdown()


def handle_beta_change(change):
    if change['new'] is None:
        return
    refresh_graph_slider()


def handle_graph_change(change):
    if suppress_graph_event:
        return
    update_plot(recompute_depth=True)


def handle_depth_change(change):
    if suppress_depth_event:
        return
    update_plot(recompute_depth=False)


file_dropdown.observe(handle_file_change, names='value')
beta_dropdown.observe(handle_beta_change, names='value')
graph_slider.observe(handle_graph_change, names='value')
depth_slider.observe(handle_depth_change, names='value')

refresh_beta_dropdown()
plot_row = HBox([gt_plot_output, pred_plot_output], layout=Layout(width='100%'))
display(VBox([file_dropdown, beta_dropdown, graph_slider, depth_slider, status_html, plot_row]))
update_plot(recompute_depth=True)